In [ ]:
import importlib
import mininn.layers
import mininn

importlib.reload(mininn.layers)
importlib.reload(mininn)

from mininn import Dropout

In [72]:
import torch
from copy import copy
from torch.utils.data import DataLoader, random_split, Subset
from torchvision import datasets, transforms

from mininn import (
    Adam,
    BatchNorm2D,
    Conv2D,
    Flatten,
    Linear,
    MaxPool2D,
    ReLU,
    ResidualBlock,
    Sequential,
    evaluate,
    load_checkpoint,
    save_checkpoint,
    train_one_epoch,
    GlobalAvgPool2D,
    Dropout
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
g = torch.Generator(device=device).manual_seed(42)

data_root = "../data_1"
batch_size = 64

In [9]:
model = Sequential([
    Conv2D(3, 8, 3, padding=1, device=device, generator=g),
    ReLU(),
    MaxPool2D(2, 2),

    Conv2D(8, 16, 3, padding=1, device=device, generator=g),
    ReLU(),
    MaxPool2D(2, 2),

    ResidualBlock(16, device=device, generator=g),
    Flatten(),
    Linear(1024, 10, device=device, generator=g),
])

images = torch.randn(
    (2, 3, 32, 32), device=device, generator=g
)
labels = torch.tensor([0, 1], device=device)

model.train()
logits = model(images)
loss = torch.nn.functional.cross_entropy(logits, labels)
loss.backward()

print("Output shape:", logits.shape)
print("Loss:", loss.item())
print("All parameter gradients present:", all(
    p.grad is not None for p in model.parameters()
))
print("All parameter gradients finite:", all(
    p.grad is not None and torch.isfinite(p.grad).all().item()
    for p in model.parameters()
))

Output shape: torch.Size([2, 10])
Loss: 7.869256973266602
All parameter gradients present: True
All parameter gradients finite: True


In [3]:
w = torch.tensor([2.0], device=device, requires_grad=True)
optimizer = Adam([w], learning_rate=0.1)

optimizer.zero_grad()
loss = w.square().sum()
loss.backward()

print("Gradient:", w.grad.item())

optimizer.step()
print("Updated weight:", w.item())

Gradient: 4.0
Updated weight: 1.899999976158142


In [40]:
# Original training dataset: no random augmentation
train_dataset = datasets.CIFAR10(
    root=data_root,
    train=True,
    download=True,
    transform=transforms.ToTensor()
)

# Fixed train/validation split
train_subset, val_subset = random_split(
    train_dataset,
    [45000, 5000],
    generator=torch.Generator().manual_seed(42)
)

# Separate dataset object so augmentation doesn't affect validation
augmented_dataset = copy(train_dataset)
augmented_dataset.transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
])

augmented_train_subset = Subset(
    augmented_dataset,
    train_subset.indices
)

# Separate test split: no random augmentation
test_dataset = datasets.CIFAR10(
    root=data_root,
    train=False,
    download=True,
    transform=transforms.ToTensor()
)

# Use this loader for parameter updates
augmented_train_loader = DataLoader(
    augmented_train_subset,
    batch_size=batch_size,
    shuffle=True
)

# Use these loaders for evaluation
train_loader = DataLoader(
    train_subset,
    batch_size=batch_size,
    shuffle=False
)

val_loader = DataLoader(
    val_subset,
    batch_size=batch_size,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

print(f"Training:   {len(train_subset):,}")
print(f"Validation: {len(val_subset):,}")
print(f"Test:       {len(test_dataset):,}")

Training:   45,000
Validation: 5,000
Test:       10,000


In [5]:
optimizer = Adam(model.parameters(), learning_rate=0.001)

train_loss = train_one_epoch(
    model, augmented_train_loader, optimizer, device
)

val_metrics = evaluate(model, val_loader, device)

print(f"Training loss: {train_loss:.4f}")
print(f"Validation loss: {val_metrics['loss']:.4f}")
print(f"Validation accuracy: {val_metrics['accuracy']:.2f}%")

Training loss: 1.5839
Validation loss: 1.6368
Validation accuracy: 46.78%


In [6]:
train_metrics = evaluate(model, train_loader, device)

print(f"Training evaluation loss: {train_metrics['loss']:.4f}")
print(f"Training evaluation accuracy: {train_metrics['accuracy']:.2f}%")

Training evaluation loss: 1.5957
Training evaluation accuracy: 47.47%


In [7]:
def take_step(w, optimizer):
    optimizer.zero_grad()
    loss = w.square().sum()
    loss.backward()
    optimizer.step()


# Train for three updates
w1 = torch.tensor([2.0], device=device, requires_grad=True)
opt1 = Adam([w1], learning_rate=0.1)

for _ in range(3):
    take_step(w1, opt1)

# Save the weight and optimizer together
saved_weight = w1.detach().clone()
saved_optimizer = opt1.state_dict()

# Recreate and restore
w2 = saved_weight.clone().requires_grad_(True)
opt2 = Adam([w2], learning_rate=0.001)
opt2.load_state_dict(saved_optimizer)

# Both take the same next step
take_step(w1, opt1)
take_step(w2, opt2)

print("Weights match:", torch.allclose(w1, w2))
print("First moments match:", torch.allclose(
    opt1.first_moments[0], opt2.first_moments[0]
))
print("Second moments match:", torch.allclose(
    opt1.second_moments[0], opt2.second_moments[0]
))
print("Step counters:", opt1.t, opt2.t)
print("Restored learning rate:", opt2.learning_rate)

Weights match: True
First moments match: True
Second moments match: True
Step counters: 4 4
Restored learning rate: 0.1


In [8]:
# Capture state before saving
state_before = model.state_dict()
step_before = optimizer.t

save_checkpoint(
    "checkpoint_helper_test.pt",
    model,
    optimizer=optimizer,
    epoch=1
)

completed_epochs = load_checkpoint(
    "checkpoint_helper_test.pt",
    model,
    optimizer=optimizer,
    device=device
)

state_after = model.state_dict()

print("Model state matches:", all(
    torch.equal(state_before[name], state_after[name])
    for name in state_before
))
print("Adam step matches:", optimizer.t == step_before)
print("Completed epochs:", completed_epochs)

Model state matches: True
Adam step matches: True
Completed epochs: 1


In [10]:
# Create once; preserve this optimizer across epochs
optimizer = Adam(model.parameters(), learning_rate=0.001)

best_val = float("-inf")

for epoch in range(1, 11):
    train_loss = train_one_epoch(
        model, augmented_train_loader, optimizer, device
    )
    metrics = evaluate(model, val_loader, device)

    if metrics["accuracy"] > best_val:
        best_val = metrics["accuracy"]
        save_checkpoint(
            "best_model.pt",
            model,
            optimizer=optimizer,
            epoch=epoch
        )

    print(
        f"Epoch {epoch} | "
        f"Train loss: {train_loss:.4f} | "
        f"Val loss: {metrics['loss']:.4f} | "
        f"Val accuracy: {metrics['accuracy']:.2f}%"
    )

Epoch 1 | Train loss: 1.5494 | Val loss: 1.4301 | Val accuracy: 48.88%
Epoch 2 | Train loss: 1.2788 | Val loss: 1.3003 | Val accuracy: 54.84%
Epoch 3 | Train loss: 1.1910 | Val loss: 1.1742 | Val accuracy: 59.26%
Epoch 4 | Train loss: 1.1375 | Val loss: 1.1456 | Val accuracy: 60.16%
Epoch 5 | Train loss: 1.0891 | Val loss: 1.1781 | Val accuracy: 58.14%
Epoch 6 | Train loss: 1.0580 | Val loss: 1.0672 | Val accuracy: 62.52%
Epoch 7 | Train loss: 1.0297 | Val loss: 1.1042 | Val accuracy: 61.62%
Epoch 8 | Train loss: 1.0042 | Val loss: 1.0941 | Val accuracy: 62.14%
Epoch 9 | Train loss: 0.9776 | Val loss: 1.0902 | Val accuracy: 62.56%
Epoch 10 | Train loss: 0.9611 | Val loss: 1.0259 | Val accuracy: 64.54%


In [3]:
projection = Conv2D(
    16, 32, kernel_size=1, stride=2,
    device=device, generator=g
)

x = torch.randn(
    (2, 16, 16, 16), device=device, generator=g
)

print("Weights:", projection.weight.shape)
print("Output:", projection(x).shape)

Weights: torch.Size([32, 16, 1, 1])
Output: torch.Size([2, 32, 8, 8])


In [ ]:
block = ResidualBlock(
    16, 32, stride=2, device=device, generator=g
)

x = torch.randn(
    (2, 16, 16, 16), device=device, generator=g
)

print("Output:", block(x).shape)
print("Shortcut keys:", [
    name for name in block.state_dict()
    if name.startswith("shortcut.")
])

Output: torch.Size([2, 32, 8, 8])
Shortcut keys: ['shortcut.weight', 'shortcut.bias']


In [5]:
optimizer_check = Adam(block.parameters(), learning_rate=0.001)
before = block.shortcut.weight.detach().clone()

optimizer_check.zero_grad()
out = block(x)
loss = out.square().mean()
loss.backward()

print("Shortcut gradient present:",
      block.shortcut.weight.grad is not None)

optimizer_check.step()

print("Shortcut weights changed:",
      not torch.equal(before, block.shortcut.weight))

Shortcut gradient present: True
Shortcut weights changed: True


In [38]:
model_projected = Sequential([
    Conv2D(3, 16, 3, padding=1, device=device, generator=g),
    BatchNorm2D(16, device=device),
    ReLU(),

    ResidualBlock(16, device=device, generator=g),
    ResidualBlock(16, 32, stride=2, device=device, generator=g),

    Flatten(),
    Linear(32 * 16 * 16, 10, device=device, generator=g),
])

In [37]:
x_check = torch.randn(
    (2, 3, 32, 32), device=device, generator=g
)

for layer in model_projected.layers:
    x_check = layer(x_check)
    print(type(layer).__name__, x_check.shape)

Conv2D torch.Size([2, 16, 32, 32])
BatchNorm2D torch.Size([2, 16, 32, 32])
ReLU torch.Size([2, 16, 32, 32])
ResidualBlock torch.Size([2, 16, 32, 32])
ResidualBlock torch.Size([2, 32, 16, 16])
Flatten torch.Size([2, 8192])
Linear torch.Size([2, 10])


In [41]:
optimizer_projected = Adam(
    model_projected.parameters(), learning_rate=0.001
)

best_val_projected = float("-inf")

for epoch in range(1, 11):
    train_loss = train_one_epoch(
        model_projected,
        augmented_train_loader,
        optimizer_projected,
        device
    )

    metrics = evaluate(model_projected, val_loader, device)

    if metrics["accuracy"] > best_val_projected:
        best_val_projected = metrics["accuracy"]
        save_checkpoint(
            "best_projected_cnn.pt",
            model_projected,
            optimizer=optimizer_projected,
            epoch=epoch
        )

    print(
        f"Epoch {epoch} | Train loss: {train_loss:.4f} | "
        f"Val loss: {metrics['loss']:.4f} | "
        f"Val accuracy: {metrics['accuracy']:.2f}%"
    )

Epoch 1 | Train loss: 1.6892 | Val loss: 1.4347 | Val accuracy: 50.46%
Epoch 2 | Train loss: 1.1971 | Val loss: 1.2169 | Val accuracy: 58.04%
Epoch 3 | Train loss: 1.0731 | Val loss: 1.2927 | Val accuracy: 57.98%
Epoch 4 | Train loss: 0.9909 | Val loss: 1.1657 | Val accuracy: 61.94%
Epoch 5 | Train loss: 0.9402 | Val loss: 1.1206 | Val accuracy: 62.32%
Epoch 6 | Train loss: 0.8973 | Val loss: 1.1012 | Val accuracy: 62.94%
Epoch 7 | Train loss: 0.8639 | Val loss: 1.0978 | Val accuracy: 63.16%
Epoch 8 | Train loss: 0.8371 | Val loss: 1.1156 | Val accuracy: 63.48%
Epoch 9 | Train loss: 0.8061 | Val loss: 1.0708 | Val accuracy: 63.64%
Epoch 10 | Train loss: 0.7850 | Val loss: 1.0872 | Val accuracy: 65.24%


In [42]:
train_metrics = evaluate(model_projected, train_loader, device)

print(f"Training evaluation loss: {train_metrics['loss']:.4f}")
print(f"Training evaluation accuracy: {train_metrics['accuracy']:.2f}%")

Training evaluation loss: 0.7265
Training evaluation accuracy: 74.88%


In [62]:
pool = GlobalAvgPool2D()

x_test = torch.tensor(
    [[[[1., 2.],
       [3., 4.]]]],
    device=device,
    requires_grad=True
)

out = pool(x_test)
out.sum().backward()

print("Output:", out)
print("Input gradients:", x_test.grad)

Output: tensor([[2.5000]], device='cuda:0', grad_fn=<MeanBackward1>)
Input gradients: tensor([[[[0.2500, 0.2500],
          [0.2500, 0.2500]]]], device='cuda:0')


In [63]:
model_gap = Sequential([
    Conv2D(3, 16, 3, padding=1, device=device, generator=g),
    BatchNorm2D(16, device=device),
    ReLU(),

    ResidualBlock(16, device=device, generator=g),
    ResidualBlock(16, 32, stride=2, device=device, generator=g),

    GlobalAvgPool2D(),
    Linear(32, 10, device=device, generator=g),
])

In [64]:
model_gap.eval()

with torch.no_grad():
    x_check = torch.randn(
        (2, 3, 32, 32), device=device, generator=g
    )

    for layer in model_gap.layers:
        x_check = layer(x_check)
        print(type(layer).__name__, x_check.shape)

Conv2D torch.Size([2, 16, 32, 32])
BatchNorm2D torch.Size([2, 16, 32, 32])
ReLU torch.Size([2, 16, 32, 32])
ResidualBlock torch.Size([2, 16, 32, 32])
ResidualBlock torch.Size([2, 32, 16, 16])
GlobalAvgPool2D torch.Size([2, 32])
Linear torch.Size([2, 10])


In [65]:
optimizer_gap = Adam(model_gap.parameters(), learning_rate=0.001)
best_val_gap = float("-inf")

for epoch in range(1, 11):
    train_loss = train_one_epoch(
        model_gap, augmented_train_loader, optimizer_gap, device
    )
    metrics = evaluate(model_gap, val_loader, device)

    if metrics["accuracy"] > best_val_gap:
        best_val_gap = metrics["accuracy"]
        save_checkpoint(
            "best_gap_cnn.pt",
            model_gap,
            optimizer=optimizer_gap,
            epoch=epoch
        )

    print(
        f"Epoch {epoch} | Train loss: {train_loss:.4f} | "
        f"Val loss: {metrics['loss']:.4f} | "
        f"Val accuracy: {metrics['accuracy']:.2f}%"
    )

# Restore the selected model and its matching optimizer state
best_epoch_gap = load_checkpoint(
    "best_gap_cnn.pt",
    model_gap,
    optimizer=optimizer_gap,
    device=device
)

train_metrics = evaluate(model_gap, train_loader, device)
val_metrics = evaluate(model_gap, val_loader, device)

print(f"\nRestored epoch: {best_epoch_gap}")
print(f"Training accuracy: {train_metrics['accuracy']:.2f}%")
print(f"Validation accuracy: {val_metrics['accuracy']:.2f}%")

Epoch 1 | Train loss: 1.6369 | Val loss: 1.5110 | Val accuracy: 44.46%
Epoch 2 | Train loss: 1.3626 | Val loss: 1.3631 | Val accuracy: 51.20%
Epoch 3 | Train loss: 1.2515 | Val loss: 1.2876 | Val accuracy: 53.28%
Epoch 4 | Train loss: 1.1830 | Val loss: 1.2964 | Val accuracy: 54.12%
Epoch 5 | Train loss: 1.1327 | Val loss: 1.2950 | Val accuracy: 54.72%
Epoch 6 | Train loss: 1.0948 | Val loss: 1.2123 | Val accuracy: 58.42%
Epoch 7 | Train loss: 1.0660 | Val loss: 1.0726 | Val accuracy: 61.76%
Epoch 8 | Train loss: 1.0381 | Val loss: 1.2212 | Val accuracy: 57.28%
Epoch 9 | Train loss: 1.0178 | Val loss: 1.0898 | Val accuracy: 61.64%
Epoch 10 | Train loss: 0.9991 | Val loss: 1.0771 | Val accuracy: 62.16%

Restored epoch: 10
Training accuracy: 63.44%
Validation accuracy: 62.16%


In [73]:
dropout = Dropout(p=0.2)
x = torch.full((10000,), 4.0, device=device)

# Training: expect roughly 20% zeros and a mean near 4
out = dropout(x)
print("Output values:", out.unique())
print("Dropped fraction:", (out == 0).float().mean().item())
print("Output mean:", out.mean().item())

# Evaluation: input passes through unchanged
dropout.training = False
print("Eval unchanged:", torch.equal(dropout(x), x))

Output values: tensor([0., 5.], device='cuda:0')
Dropped fraction: 0.19929999113082886
Output mean: 4.003499984741211
Eval unchanged: True


In [74]:
model_dropout = Sequential([
    Conv2D(3, 16, 3, padding=1, device=device, generator=g),
    BatchNorm2D(16, device=device),
    ReLU(),

    ResidualBlock(16, device=device, generator=g),
    ResidualBlock(16, 32, stride=2, device=device, generator=g),

    Flatten(),
    Dropout(p=0.2),
    Linear(8192, 10, device=device, generator=g),
])

In [75]:
optimizer_dropout = Adam(
    model_dropout.parameters(), learning_rate=0.001
)
best_val_dropout = float("-inf")

for epoch in range(1, 11):
    train_loss = train_one_epoch(
        model_dropout, augmented_train_loader,
        optimizer_dropout, device
    )
    metrics = evaluate(model_dropout, val_loader, device)

    if metrics["accuracy"] > best_val_dropout:
        best_val_dropout = metrics["accuracy"]
        save_checkpoint(
            "best_dropout_cnn.pt",
            model_dropout,
            optimizer=optimizer_dropout,
            epoch=epoch
        )

    print(
        f"Epoch {epoch} | Train loss: {train_loss:.4f} | "
        f"Val loss: {metrics['loss']:.4f} | "
        f"Val accuracy: {metrics['accuracy']:.2f}%"
    )

best_epoch = load_checkpoint(
    "best_dropout_cnn.pt",
    model_dropout,
    optimizer=optimizer_dropout,
    device=device
)

train_metrics = evaluate(model_dropout, train_loader, device)
val_metrics = evaluate(model_dropout, val_loader, device)

print(f"\nRestored epoch: {best_epoch}")
print(f"Training accuracy: {train_metrics['accuracy']:.2f}%")
print(f"Validation accuracy: {val_metrics['accuracy']:.2f}%")

Epoch 1 | Train loss: 1.7261 | Val loss: 1.3448 | Val accuracy: 53.12%
Epoch 2 | Train loss: 1.2729 | Val loss: 1.2460 | Val accuracy: 56.76%
Epoch 3 | Train loss: 1.1472 | Val loss: 1.1930 | Val accuracy: 58.62%
Epoch 4 | Train loss: 1.0822 | Val loss: 1.1859 | Val accuracy: 59.50%
Epoch 5 | Train loss: 1.0281 | Val loss: 1.2232 | Val accuracy: 59.86%
Epoch 6 | Train loss: 0.9843 | Val loss: 1.1241 | Val accuracy: 62.54%
Epoch 7 | Train loss: 0.9608 | Val loss: 1.1193 | Val accuracy: 63.04%
Epoch 8 | Train loss: 0.9280 | Val loss: 1.1493 | Val accuracy: 63.06%
Epoch 9 | Train loss: 0.8942 | Val loss: 1.0914 | Val accuracy: 63.50%
Epoch 10 | Train loss: 0.8877 | Val loss: 1.0807 | Val accuracy: 64.80%

Restored epoch: 10
Training accuracy: 71.80%
Validation accuracy: 64.80%
